In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import os
for dirname, _, filenames in os.walk('raw'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

raw\data_dictionary_trip_records_yellow.pdf
raw\taxi_zone_lookup.csv
raw\yellow_tripdata_2026-01.parquet
raw\taxi_zones\taxi_zones.cpg
raw\taxi_zones\taxi_zones.dbf
raw\taxi_zones\taxi_zones.prj
raw\taxi_zones\taxi_zones.shp
raw\taxi_zones\taxi_zones.shx


In [3]:
df = pd.read_parquet('raw\\yellow_tripdata_2026-01.parquet').sample(frac=2/5, random_state=42)
taxizone_lookup = pd.read_csv('raw\\taxi_zone_lookup.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1489956 entries, 1511419 to 191999
Data columns (total 20 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   VendorID               1489956 non-null  int32         
 1   tpep_pickup_datetime   1489956 non-null  datetime64[us]
 2   tpep_dropoff_datetime  1489956 non-null  datetime64[us]
 3   passenger_count        1054108 non-null  float64       
 4   trip_distance          1489956 non-null  float64       
 5   RatecodeID             1054108 non-null  float64       
 6   store_and_fwd_flag     1054108 non-null  object        
 7   PULocationID           1489956 non-null  int32         
 8   DOLocationID           1489956 non-null  int32         
 9   payment_type           1489956 non-null  int64         
 10  fare_amount            1489956 non-null  float64       
 11  extra                  1489956 non-null  float64       
 12  mta_tax                14899

In [5]:
taxizone_lookup.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   LocationID    265 non-null    int64 
 1   Borough       264 non-null    object
 2   Zone          264 non-null    object
 3   service_zone  263 non-null    object
dtypes: int64(1), object(3)
memory usage: 8.4+ KB


In [6]:
taxizone_lookup

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
...,...,...,...,...
260,261,Manhattan,World Trade Center,Yellow Zone
261,262,Manhattan,Yorkville East,Yellow Zone
262,263,Manhattan,Yorkville West,Yellow Zone
263,264,Unknown,NaN,NaN


## Predecir el precio

### Selección de variables

In [7]:
df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='object')

In [8]:
df = df.loc[:,['tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount']].reset_index(drop=True)

#### Columnas eliminadas debido a evidente riesgo de dataleakage, o por representar tarifas adicionales que solo calculan al final del viaje

- tip_amount
- total_amount
- tolls_amount
- mta_tax
- extra
- improvement_surcharge
- congestion_surcharge
- Airport_fee
- cbd_congestion_fee

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1489956 entries, 0 to 1489955
Data columns (total 9 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   tpep_pickup_datetime   1489956 non-null  datetime64[us]
 1   tpep_dropoff_datetime  1489956 non-null  datetime64[us]
 2   passenger_count        1054108 non-null  float64       
 3   trip_distance          1489956 non-null  float64       
 4   RatecodeID             1054108 non-null  float64       
 5   PULocationID           1489956 non-null  int32         
 6   DOLocationID           1489956 non-null  int32         
 7   payment_type           1489956 non-null  int64         
 8   fare_amount            1489956 non-null  float64       
dtypes: datetime64[us](2), float64(4), int32(2), int64(1)
memory usage: 90.9 MB


## Limpieza de datos

In [10]:
df["passenger_count"] = df["passenger_count"].fillna(0).astype(int)

#Seleccionar solo datos válidos de trip_distance y fare_amount
df = df[(df["trip_distance"] > 0) & (df["fare_amount"] > 0) & (df["passenger_count"] > 0)]
#Valides por fechas dropoff < pickup
df = df[df["tpep_dropoff_datetime"] > df["tpep_pickup_datetime"]]

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1002542 entries, 0 to 1489955
Data columns (total 9 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   tpep_pickup_datetime   1002542 non-null  datetime64[us]
 1   tpep_dropoff_datetime  1002542 non-null  datetime64[us]
 2   passenger_count        1002542 non-null  int32         
 3   trip_distance          1002542 non-null  float64       
 4   RatecodeID             1002542 non-null  float64       
 5   PULocationID           1002542 non-null  int32         
 6   DOLocationID           1002542 non-null  int32         
 7   payment_type           1002542 non-null  int64         
 8   fare_amount            1002542 non-null  float64       
dtypes: datetime64[us](2), float64(3), int32(3), int64(1)
memory usage: 65.0 MB
